# 05 - Execution-based metrics

The only metrics that measure semantics on live data, and the strongest correctness signal in the evaluation. The gold SQL runs on the PostgreSQL oracle; the generated query runs on its graph database (Cypher on Neo4j, AQL on ArangoDB, Gremlin on Gremlin Server); the two row multisets are normalised and compared. Execution accuracy (EX) is exact multiset equality; result precision, recall, and F1 refine it (these measure row-multiset overlap, and are distinct from the structural Component F1 of notebook 03). Caveats baked into the comparator: a 180 s per-query timeout (a slow-but-correct query still scores 0 if killed), date reconciliation to epoch-millis across the stores, empty string mapped to NULL for absent optional text on AQL and Gremlin, and a vacuous-pass blind spot (both sides empty gives EX 1.0). The comparator and per-backend runners are harness primitives; the per-record loop lives here.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / "pyproject.toml").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / "eval"))

import json
import os

import matplotlib.pyplot as plt
import pandas as pd
from harness import (
    EXECUTION_CACHE_PATH,
    FIGURES_DIR,
    METRICS_EXECUTION_CSV,
    RECORDS_DIR,
    frames,
    load_records,
    plots,
)
from harness import execution as ex

OUT_CSV = METRICS_EXECUTION_CSV
CACHE_PATH = EXECUTION_CACHE_PATH
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

## Helper methods

`execute_records` is the per-record engine (previously hidden in the harness, surfaced here so the execution flow reads in the notebook): it caches the Postgres oracle rows on disk keyed by `dataset:query_id`, skips translations that failed validation, and calls the harness primitives `run_postgres`, `RUNNERS[target]`, and `compare_rowsets`. Backends cannot all run at once, so `EVAL_EXECUTION_TARGETS` selects which targets execute this pass; the save cell preserves the other targets' rows.

In [ ]:
def execute_records(records, target, cache, cache_path):
    """Execute every runnable record of one target; return the metric rows as a DataFrame.

    Postgres oracle rows are cached in `cache` (persisted to `cache_path`, keyed dataset:query_id),
    so re-runs and other targets touch nothing but their graph DB. Records that failed validation
    (or errored) score 0 without touching the backend.
    """
    runnable = [r for r in records if r["dataset"] in ex.POSTGRES_DATASETS and r["target"] == target]
    print(f"{target}: {len(runnable)} executable record(s).")
    out_rows = []
    for i, rec in enumerate(runnable, 1):
        qid = rec["query_id"]
        ckey = f"{rec['dataset']}:{qid}"
        print(f"[{i:3d}/{len(runnable)}] {qid} ({target})", end=" ", flush=True)
        if ckey in cache:
            ref_rows = [tuple(r) for r in cache[ckey]["rows"]]
            ref_runtime = cache[ckey]["runtime"]
            ref_error = cache[ckey].get("error")
        else:
            ref_rows, ref_runtime, ref_error = ex.run_postgres(rec["sql"])
            cache[ckey] = {"rows": [list(r) for r in ref_rows], "runtime": ref_runtime,
                           "error": ref_error}
            cache_path.write_text(json.dumps(cache, default=str))
        row = {"dataset": rec["dataset"], "target": target, "model": rec["model"], "query_id": qid,
               "difficulty": rec["difficulty"], "validation_passed": rec["validation_passed"],
               "reference_runtime_s": ref_runtime, "reference_error": ref_error,
               "translated_runtime_s": None, "execution_error": None,
               "execution_accuracy": 0.0, "result_precision": 0.0, "result_recall": 0.0,
               "result_f1": 0.0, "result_jaccard_dist": 1.0,
               "reference_rows": len(ref_rows), "translated_rows": 0}
        if ref_error is not None:
            print(f"REF ERROR ({ref_error})")
            out_rows.append(row)
            continue
        if not rec["validation_passed"] or not rec.get("generated_query"):
            print("skip (translation invalid)")
            out_rows.append(row)
            continue
        trans_rows, trans_runtime, trans_error = ex.RUNNERS[target](rec["generated_query"])
        row["translated_runtime_s"] = trans_runtime
        row["execution_error"] = trans_error
        if trans_error is not None:
            print(f"EXEC ERROR ({trans_error[:60]})")
            out_rows.append(row)
            continue
        row.update(ex.compare_rowsets(ref_rows, trans_rows, empty_as_null=target in ex.EMPTY_AS_NULL_TARGETS))
        marker = "ok" if row["execution_accuracy"] == 1.0 else "ne"
        print(f"{marker} EX={row['execution_accuracy']:.0f} F1={row['result_f1']:.2f} "
              f"ref={row['reference_rows']} trans={row['translated_rows']}")
        out_rows.append(row)
    return pd.DataFrame(out_rows)

In [ ]:
EXEC_TARGETS = [t.strip() for t in os.environ.get("EVAL_EXECUTION_TARGETS", "cypher,aql,gremlin").split(",") if t.strip()]
records = load_records(RECORDS_DIR)
cache = json.loads(CACHE_PATH.read_text()) if CACHE_PATH.exists() else {}
print(f"Loaded {len(records)} record(s); reference cache: {len(cache)} entries; executing: {EXEC_TARGETS}")

FEATURES = frames.feature_map("ldbc")
EXEC_COLS = ["query_id", "difficulty", "execution_accuracy", "result_precision", "result_recall",
             "result_f1", "result_jaccard_dist", "execution_error", "translated_runtime_s",
             "reference_rows", "translated_rows"]
EXEC_METRIC_COLS = ["execution_accuracy", "result_precision", "result_recall", "result_f1", "result_jaccard_dist"]


def execute_model(target, model, bucket):
    """Execute one model's queries for `target` (None if target skipped this pass); stash rows in bucket."""
    if target not in EXEC_TARGETS:
        print(f"{target}: skipped (not in EVAL_EXECUTION_TARGETS); existing rows preserved.")
        return None
    frame = execute_records([r for r in records if r["model"] == model], target, cache, CACHE_PATH)
    if not len(frame):
        print(f"No executed records for {model}.")
        return None
    bucket.append(frame)
    return frame.sort_values("query_id")[EXEC_COLS].reset_index(drop=True)


# summary_by_model / by_feature are thin wrappers over harness.frames (execution has its own
# metric-column set); execute_records above stays in the notebook as the visible execution flow.
def summary_by_model(frame):
    """Mean execution metrics by model (canonical order); frame = one target's rows."""
    if frame is None or not len(frame):
        print("Target skipped / empty this pass.")
        return None
    return frames.order_frame_models(frame).groupby("model", observed=True)[EXEC_METRIC_COLS].mean()


def by_feature(frame, cols):
    """Mean of `cols` per (target, SQL feature label)."""
    return frames.by_feature(frame, cols, FEATURES)

## Prerequisites

The execution databases must be reachable (only the targets in `EVAL_EXECUTION_TARGETS`). At most one graph DB is up at a time on this host; a warning here is fine for the targets you are not executing this pass.

In [ ]:
print(f"Backends: {ex.TARGET_DB} | per-query timeout: {ex.TIMEOUT_S}s | executing: {EXEC_TARGETS}")


def _warn_db(name, err, hint=""):
    if err:
        print(f"  {name}: unavailable ({(err.splitlines() or [''])[0][:90]})")
    elif hint:
        print(f"  {name}: reachable, but {hint}")
    else:
        print(f"  {name}: reachable")


print("Execution databases:")
_warn_db("Postgres (5433)", ex.run_postgres("SELECT 1")[2])
if "cypher" in EXEC_TARGETS:
    _warn_db("Neo4j (7687)", ex.run_cypher("RETURN 1 AS ok")[2])
if "aql" in EXEC_TARGETS:
    aql_err = ex.run_aql("RETURN 1")[2]
    # run_aql expands KNOWS -> graphonauts's split `knows` collection, so this probes whether
    # LDBC is loaded (no unified-edge build step exists any more; see harness.arango_edges).
    hint = "graphonauts split edge collections not loaded (load LDBC into ArangoDB db 'graphonauts')" \
        if (not aql_err and ex.run_aql("FOR e IN KNOWS LIMIT 1 RETURN 1")[2]) else ""
    _warn_db("ArangoDB (8529, graphonauts)", aql_err, hint)
if "gremlin" in EXEC_TARGETS:
    _warn_db("Gremlin (8182)", ex.run_gremlin("g.V().limit(1)")[2])

## SQL to Cypher

In [ ]:
cypher_frames = []  # per-model execution rows for this target

### llama3.2:latest

In [ ]:
execute_model('cypher', 'llama3.2:latest', cypher_frames)

### qwen3-coder:30b

In [ ]:
execute_model('cypher', 'qwen3-coder:30b', cypher_frames)

### gemma4:26b

In [ ]:
execute_model('cypher', 'gemma4:26b', cypher_frames)

### claude-opus-4-8

In [ ]:
execute_model('cypher', 'claude-opus-4-8', cypher_frames)

### claude-opus-4-8-thinking

In [ ]:
execute_model('cypher', 'claude-opus-4-8-thinking', cypher_frames)

### Aggregation by model

In [ ]:
cypher_exec_df = pd.concat(cypher_frames, ignore_index=True) if cypher_frames else None
summary_by_model(cypher_exec_df)

### Figures

In [ ]:
sub = cypher_exec_df
label = 'SQL -> Cypher'
prefix = 'cypher'

#### Execution accuracy by query x model

In [ ]:
if sub is not None and len(sub):
    p = FIGURES_DIR / f"{prefix}_query_model_exec.png"
    plots.query_model_heatmap(sub, "execution_accuracy", p, discrete=True, title=f"{label}: execution accuracy by query x model", cbar_label="exec acc")
    plots.show(p)
    plt.close("all")
else:
    print(f"{label}: not executed this pass.")

#### Execution accuracy and result F1 per model

In [ ]:
if sub is not None and len(sub):
    p = FIGURES_DIR / f"{prefix}_execution_bars.png"
    plots.per_model_bars(sub, ["execution_accuracy", "result_f1"], p, title=f"{label}: execution accuracy and result F1 per model", ylabel="score", labels={"execution_accuracy": "exec accuracy", "result_f1": "result F1"})
    plots.show(p)
    plt.close("all")
else:
    print(f"{label}: not executed this pass.")

## SQL to AQL

In [ ]:
aql_frames = []  # per-model execution rows for this target

### llama3.2:latest

In [ ]:
execute_model('aql', 'llama3.2:latest', aql_frames)

### qwen3-coder:30b

In [ ]:
execute_model('aql', 'qwen3-coder:30b', aql_frames)

### gemma4:26b

In [ ]:
execute_model('aql', 'gemma4:26b', aql_frames)

### claude-opus-4-8

In [ ]:
execute_model('aql', 'claude-opus-4-8', aql_frames)

### claude-opus-4-8-thinking

In [ ]:
execute_model('aql', 'claude-opus-4-8-thinking', aql_frames)

### Aggregation by model

In [ ]:
aql_exec_df = pd.concat(aql_frames, ignore_index=True) if aql_frames else None
summary_by_model(aql_exec_df)

### Figures

In [ ]:
sub = aql_exec_df
label = 'SQL -> AQL'
prefix = 'aql'

#### Execution accuracy by query x model

In [ ]:
if sub is not None and len(sub):
    p = FIGURES_DIR / f"{prefix}_query_model_exec.png"
    plots.query_model_heatmap(sub, "execution_accuracy", p, discrete=True, title=f"{label}: execution accuracy by query x model", cbar_label="exec acc")
    plots.show(p)
    plt.close("all")
else:
    print(f"{label}: not executed this pass.")

#### Execution accuracy and result F1 per model

In [ ]:
if sub is not None and len(sub):
    p = FIGURES_DIR / f"{prefix}_execution_bars.png"
    plots.per_model_bars(sub, ["execution_accuracy", "result_f1"], p, title=f"{label}: execution accuracy and result F1 per model", ylabel="score", labels={"execution_accuracy": "exec accuracy", "result_f1": "result F1"})
    plots.show(p)
    plt.close("all")
else:
    print(f"{label}: not executed this pass.")

## SQL to Gremlin

In [ ]:
gremlin_frames = []  # per-model execution rows for this target

### llama3.2:latest

In [ ]:
execute_model('gremlin', 'llama3.2:latest', gremlin_frames)

### qwen3-coder:30b

In [ ]:
execute_model('gremlin', 'qwen3-coder:30b', gremlin_frames)

### gemma4:26b

In [ ]:
execute_model('gremlin', 'gemma4:26b', gremlin_frames)

### claude-opus-4-8

In [ ]:
execute_model('gremlin', 'claude-opus-4-8', gremlin_frames)

### claude-opus-4-8-thinking

In [ ]:
execute_model('gremlin', 'claude-opus-4-8-thinking', gremlin_frames)

### Aggregation by model

In [ ]:
gremlin_exec_df = pd.concat(gremlin_frames, ignore_index=True) if gremlin_frames else None
summary_by_model(gremlin_exec_df)

### Figures

In [ ]:
sub = gremlin_exec_df
label = 'SQL -> Gremlin'
prefix = 'gremlin'

#### Execution accuracy by query x model

In [ ]:
if sub is not None and len(sub):
    p = FIGURES_DIR / f"{prefix}_query_model_exec.png"
    plots.query_model_heatmap(sub, "execution_accuracy", p, discrete=True, title=f"{label}: execution accuracy by query x model", cbar_label="exec acc")
    plots.show(p)
    plt.close("all")
else:
    print(f"{label}: not executed this pass.")

#### Execution accuracy and result F1 per model

In [ ]:
if sub is not None and len(sub):
    p = FIGURES_DIR / f"{prefix}_execution_bars.png"
    plots.per_model_bars(sub, ["execution_accuracy", "result_f1"], p, title=f"{label}: execution accuracy and result F1 per model", ylabel="score", labels={"execution_accuracy": "exec accuracy", "result_f1": "result F1"})
    plots.show(p)
    plt.close("all")
else:
    print(f"{label}: not executed this pass.")

## Run-level summary and save

Merge this pass's targets into the CSV, preserving the rows of any target not executed now (backends cannot all run at once).

In [ ]:
new_frames = {t: f for t, f in (("cypher", cypher_exec_df), ("aql", aql_exec_df),
                                ("gremlin", gremlin_exec_df)) if f is not None}
if OUT_CSV.exists():
    exec_all = pd.read_csv(OUT_CSV)
    exec_all = exec_all[~exec_all["target"].isin(new_frames)]
else:
    exec_all = pd.DataFrame()
to_concat = ([exec_all] if len(exec_all) else []) + [f for f in new_frames.values() if len(f)]
exec_all = pd.concat(to_concat, ignore_index=True) if to_concat else pd.DataFrame()
if len(exec_all):
    e = frames.order_frame_models(exec_all)
    print("By target x model:")
    display(e.groupby(["target", "model"], observed=True)[EXEC_METRIC_COLS].mean())
    print("By target x SQL feature:")
    display(by_feature(exec_all, EXEC_METRIC_COLS))
exec_all.to_csv(OUT_CSV, index=False)
print(f"Wrote {len(exec_all)} rows to {OUT_CSV} (executed this pass: {', '.join(new_frames) or 'none'}).")
ex.close_clients()

## Summary

Execution is where the models, and the target languages, separate. On the declarative targets precision, recall, and F1 track EX closely (failures are all-or-nothing). Gremlin roughly halves the strongest configurations and crushes the weakest; how far extended thinking recovers that gap is the evaluation's headline, dissected in notebook 06 and the thesis.

Wrote `metrics_execution.csv`. Proceed to **`06_report.ipynb`**.